<a href="https://colab.research.google.com/github/psiperez/budas_trade/blob/main/barra1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def calculate_backtest_barra1():
    print("=" * 70)
    print("BACKTEST BARRA1 - RESULTADOS")
    print("=" * 70)

    # 1. Download Data - Ajustado para os últimos 60 dias para garantir disponibilidade de dados 1h
    end_date = datetime.now()
    start_date = end_date - timedelta(days=58)

    data_1h = yf.download('GC=F', start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='15m', progress=False)

    if data_1h.empty:
        return "Dados não encontrados. Tente um ativo diferente ou um período mais curto."

    if isinstance(data_1h.columns, pd.MultiIndex):
        data_1h.columns = data_1h.columns.get_level_values(0)

    # 2. Preparar dados simulados de M15
    df_list = []
    prev_close = data_1h.iloc[0]['Open']
    for idx, row in data_1h.iterrows():
        for i in range(4):
            factor = (i + 1) / 4.0
            candle_time = idx + timedelta(minutes=15 * i)
            candle_open = prev_close
            range_p = float(row['High']) - float(row['Low'])
            candle_high = float(row['Open']) + range_p * (factor * 0.8 + np.random.uniform(0, 0.2))
            candle_low = float(row['Open']) - range_p * (0.2 + np.random.uniform(0, 0.2))
            candle_high = max(float(candle_high), float(candle_low) + 0.1)
            candle_close = float(row['Open']) + range_p * factor if i < 3 else float(row['Close'])
            prev_close = candle_close
            df_list.append({'dt': candle_time, 'Open': candle_open, 'High': candle_high, 'Low': candle_low, 'Close': candle_close})

    df = pd.DataFrame(df_list).set_index('dt')

    # 3. Simulação simplificada
    balance = 100000.0
    trades = []
    positions = []

    for idx, row in df.iterrows():
        for p in positions[:]:
            hit_sl = (p['dir'] == 1 and row['Low'] <= p['sl']) or (p['dir'] == -1 and row['High'] >= p['sl'])
            hit_tp = (p['dir'] == 1 and row['High'] >= p['tp']) or (p['dir'] == -1 and row['Low'] <= p['tp'])
            if hit_sl or hit_tp:
                res = (p['sl'] if hit_sl else p['tp']) - p['entry']
                profit = res * p['dir'] * p['lot'] * 100
                balance += profit
                trades.append(profit)
                positions.remove(p)

        if len(positions) == 0 and np.random.random() < 0.05:
            dir = 1 if row['Close'] > row['Open'] else -1
            entry = row['Close']
            sl = entry - (5 * dir)
            tp = entry + (10 * dir)
            positions.append({'entry': entry, 'sl': sl, 'tp': tp, 'dir': dir, 'lot': 1.0})

    print(f"\nPeríodo: {start_date.date()} até {end_date.date()}")
    print(f"Saldo Final: ${balance:,.2f}")
    print(f"Total de Trades: {len(trades)}")
    print(f"Lucro Líquido: ${sum(trades):,.2f}")

calculate_backtest_barra1()

BACKTEST BARRA1 - RESULTADOS


/tmp/ipykernel_5785/1456228476.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_1h = yf.download('GC=F', start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='15m', progress=False)



Período: 2026-01-18 até 2026-03-17
Saldo Final: $-199,500.00
Total de Trades: 668
Lucro Líquido: $-299,500.00


Você é um especialista na linguagem de programação mql. Acesse o repositório psiperez/budas_trade, crie um arquivo novo com o nome de mar3_eurusd.mq5, a partir do arquivo mar3_agressivo.mq5, para operar de forma lucrativa no par eurusd, considerando realizar entradas contrárias ao filtro de tendência EMA.

Acesse o repositório psiperez/budas_trade, crie um arquivo novo com o nome de mar2.1_agressivo_invertido.mq5, a partir do arquivo mar2.1_xauusd_agressivo.mq5, e faça as seguintes alterações : 1) inverta a lógica de tendência na função PlaceOrders. Agora o robô realiza compras quando o preço está abaixo da EMA e vendas quando está acima (Mean Reversion Breakout).2) Estabeleça realizações parciais de lucro nos patamares de 25%, 50%, 75% e 100% do take profit total. 3) verifique a lógica do trail stop dinâmico e do ATR para evitar conflitos e, se houver conflitos, corrija.

Acesse o repositório psiperez/budas_trade, crie um arquivo novo com o nome de mar2.1_agressivo_invertido.mq5, a partir do arquivo mar2.1_xauusd_agressivo.mq5, e faça as seguintes alterações : 1) inverta a lógica de tendência na função PlaceOrders. Agora o robô realiza compras quando o preço está abaixo da EMA e vendas quando está acima (Mean Reversion Breakout).2) Estabeleça realizações parciais de lucro nos patamares de 25%, 50%, 75% e 100% do take profit total.

Acesse o repositório psiperez/budas_trade, crie um arquivo novo com o nome de mar2.1_agressivo_invertido2.mq5, a partir do arquivo mar2.1_agressivo_invertido.mq5, e faça as seguintes alterações : 1)NA função PlaceBuy e PlaceSell, use ordens BuyLimit e SellLimit, ajustando os seus respectivos Take Profit e Stop Loss de forma coerente com as modalidades BuyLimit e SellLimit

No repositório psiperez/budas_trade altere o arquivo read me, que passará a ter a seguinte descrição a seguir :

Visão Geral da Estratégia

A estratégia é baseada na análise de um indicador de tendência (ATR Envelope) para determinar momentos de entrada e saída no mercado, gerenciando as posições com stops, take profits, parcializações e trailing stops, visando maximizar lucros e limitar perdas.

Resumo Operacional

Quando entra: ao detectar mudança de tendência usando o ATR Envelope, entra comprado ou vendido.
Como gerencia: com stops, take profits, parcializações, break-even e trailing stops.
Objetivo: capturar movimentos de tendência, protegendo lucros e limitando perdas. Resultado esperado: ganho consistente em mercados de tendência, com gerenciamento ativo de risco.

Detalhamento Operacional : O Indicador de Tendência (ATR Envelope) calcula uma faixa dinâmica ao redor do preço, usando o ATR (Average True Range). Essa faixa é composta por uma linha superior (Smax) e uma inferior (Smin), ajustadas com base na volatilidade e um fator de desvio. A tendência de mercado é identificada com base na relação do preço de fechamento (Close) com essas linhas: Se o preço ultrapassa Smax, indica tendência de alta (trend=1). Se o preço cai abaixo de Smin, indica tendência de baixa (trend=-1). Caso contrário, mantém a tendência anterior.
Geração de Sinais de Entrada: Quando há uma mudança de tendência (por exemplo, de baixa para alta ou vice-versa), o sistema: Fecha posições abertas. Abre uma nova posição na direção da tendência detectada. Define o nível de stop loss (SL) baseado na última linha de suporte ou resistência (Smax ou Smin). Calcula o tamanho da posição (lot_size) proporcional ao risco definido (1% do saldo).
Gestão das Posições: Para cada posição aberta, o sistema monitora continuamente:
Stop Loss (SL): se atingido, a posição é fechada com prejuízo.
Take Profit (TP): se atingido, a posição é fechada com lucro.
Parciais: realiza parcializações ao atingir 25%, 50% e 75% do lucro potencial, reduzindo o lote e ajustando o SL.
Break-even: ao atingir um lucro equivalente a uma vez o ATR, ajusta o SL para o preço de entrada, protegendo lucros.
Trailing Stop: ajusta o SL em direção à direção do movimento para maximizar ganhos.
Gerenciamento de Risco e Tamanho de Posição: O tamanho da posição é calculado com base no risco fixo (1%) do saldo total, e na distância do SL.

Você é um especialista na linguagem de programação mql. Acesse o repositório psiperez/budas trade e explique quais causas podem estar levando o EA envelope1 a não permanecer ativo no gráfico do metatrader 5

Reescreva o código do EA envelope1 com a implementação das correções necessárias para sanar as vulnerabilidades contidas no código e descritas a seguir :
1)Há dois inputs diferentes para o mesmo propósito (ATR_Period e inpAtrPeriod). No OnInit() você usa ATR_Period para criar o indicador, mas no cálculo manual do envelope usa inpAtrPeriod. Isso pode causar inconsistência nos sinais.
2)Está sendo usando um índice fixo (rates_total - 1 - 2) para acessar o buffer, mas o array workTrendEnvelopes é redimensionado com base em bars. Isso pode acessar memória inválida.
3)É copiado apenas 1 valor do ATR, mas em ManagePosition() usa CachedATR para várias posições simultâneas. Se houver múltiplas posições, todas usarão o mesmo valor de ATR, o que pode não ser adequado.
4)Há possibilidade de ocorrer o loop infinito no processamento de candles do lookback
5)O SeriesInfoInteger() deve ser substituído por Bars() para obter o total de barras
6)O EA não verifica se já existe uma posição aberta antes de abrir uma nova na mudança de tendência. Se res_curr.trendChange for verdadeiro, ele abre nova posição mesmo com posição existente.
7)Se o preço atingir o break-even antes do TP parcial, as lógicas do Break-even e do Trailing podem conflitar.


Acesse o repositório psiperez/budas_trade, crie um arquivo novo com o nome de barra1.mq5, a partir do arquivo envelope1.mq5, e altere a lógica do código a partir da substituição do indicador ATR TREND ENVELOPES pelas Bandas de Bollinger, que passa a funcionar com a seguinte regra operacional :
1) Uma ordem Sell Stop deve ser aberta quando no sentido inverso

Acesse o repositório psiperez/budas_trade, crie um arquivo novo com o nome de barra1.mq5, cópia a lógica de programação do arquivo envelope1.mq5, e aproveite as instruções que forem compatíveis com a seguinte regra operacional:
No timeframe de 15 minutos, ordens devem ser abertas somente após 10 minutos de início de cada candle, considerando os percentuais de retração Fibonacci calculados a partir valor de máxima e de mínima do candle do seguinte modo :
1) Se for um candle de venda (preço de abertura maior que preço aos 10 minutos), deve ser aberta uma ordem de venda pendente  tipo stop e uma ordem de venda pendente tipo limit no valor de 38.2% da retração de Fibonacci com stop loss na linha de 50% de Fibonacci e take profit no valor de fechamento do candle
2) Se for um candle de compra (preço de abertura menor que preço aos 10 minutos), deve ser aberta uma ordem de compra pendente  tipo stop e uma ordem de venda pendente tipo limit no valor de 38.2% da retração de Fibonacci com stop loss na linha de 50% de Fibonacci e take profit no valor de fechamento do candle

Reescreva o código do EA Envelope1.mq5 para seguir a estratégia operacional a seguir. A partir da verificação de mudança de tendência no ATR Trend Envelope, trace um canal equidistante e uma retração fibonacci de 0 a 100% entre os dois limites do envelope para a abertura de um posição, por meio de uma ordem pendente do tipo stop, no nível de 50% da fibonacci dentro do canal equidistante no sentido da tendência